# Reactant.jl integration

Reactant.jl is a Julia frontend to the MLIR framework and XLA compiler. It takes Julia code, optimizes aggresively and compiles to CPU / GPU / TPU / distributed clusters.

In [11]:
using Tangles
using Reactant
using Adapt
using Random
using Chairmarks

In [12]:
Random.seed!(1234)
tn = rand(SimpleTensorNetwork, 4, 3)

SimpleTensorNetwork (#tensors=4, #inds=6)

Reactant.jl traces the use of its own array types; i.e. it only tracks the operations performed on `Reactant.ConcreteRArray`s.

So first, the arrays of tensors in the Tensor Network need to be converted to `ConcreteRArray`. This is easily done with `Adapt.adapt`.

In [13]:
tnre = adapt(ConcreteRArray, tn)

SimpleTensorNetwork (#tensors=4, #inds=6)

In [14]:
@info "typeof(tensors(tn)[1])" before = typeof(tensors(tn)[1]) after = typeof(tensors(tnre)[1])

┌ Info: typeof(tensors(tn)[1])
│   before = NamedTensor{Float64, 6, Array{Float64, 6}}
│   after = NamedTensor{Float64, 6, ConcretePJRTArray{Float64, 6, 1}}
└ @ Main /Users/mofeing/Developer/Tangles.jl/examples/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sZmlsZQ==.jl:1


`Reactant.@compile` compiles the given code using the compilers mentioned above and returns a _thunk_ (a _functor_, a callable object). The thunk accepts the same type of arguments as the ones passed to `Reactant.@compile`.

Note that `sync=true` is required for benchmarking, as compiled functions might execute asynchronously.

In [15]:
f = @compile sync=true contract(tnre)
f(tnre)

0-dimensional NamedTensor(::Tensor(::ConcretePJRTArray{Float64,0})):
1291.8407836130243

`@jit` is a helper macro that is equivalent to `@compile` and then calling the compiled function.

In [16]:
@jit contract(tnre)

0-dimensional NamedTensor(::Tensor(::ConcretePJRTArray{Float64,0})):
1291.8407836130243

One important aspect is that since allocations are being performed on the "C-side", the allocations shown when profiling are not really representative of the real allocations (only of the allocations being performed on the Julia side).

The time though, should be ok (unless you use asynchronous execution).

In [17]:
@b f(tnre)

71.591 μs (38 allocs: 992 bytes)

## Inspecting what's happenning under the hood

Just like with Julia's `@code_lowered`, `@code_llvm`, `@code_native`, ... introspection tools, Reactant.jl provides the `@code_hlo`, `@code_mhlo` and `@code_xla` tools to inspect the emitted MLIR and HLO code.

Despite its name, `@code_hlo` prints the MLIR representation of the code, which is the main representation Reactant.jl works with. For example, the MLIR code of the compiled contraction is:

In [18]:
@code_hlo contract(tnre)

module @reactant_contract attributes {mhlo.num_partitions = 1 : i64, mhlo.num_replicas = 1 : i64} {
  func.func @main(%arg0: tensor<7x6x7x8x5x2xf64> {enzymexla.memory_effects = []}, %arg1: tensor<6x7xf64> {enzymexla.memory_effects = []}, %arg2: tensor<2x7xf64> {enzymexla.memory_effects = []}, %arg3: tensor<5x8xf64> {enzymexla.memory_effects = []}) -> tensor<f64> attributes {enzymexla.memory_effects = []} {
    %0 = stablehlo.transpose %arg0, dims = [5, 4, 3, 2, 1, 0] : (tensor<7x6x7x8x5x2xf64>) -> tensor<2x5x8x7x6x7xf64>
    %1 = stablehlo.dot_general %0, %arg1, contracting_dims = [3, 4] x [1, 0], precision = [DEFAULT, DEFAULT] : (tensor<2x5x8x7x6x7xf64>, tensor<6x7xf64>) -> tensor<2x5x8x7xf64>
    %2 = stablehlo.dot_general %1, %arg3, contracting_dims = [1, 2] x [0, 1], precision = [DEFAULT, DEFAULT] : (tensor<2x5x8x7xf64>, tensor<5x8xf64>) -> tensor<2x7xf64>
    %3 = stablehlo.dot_general %arg2, %2, contracting_dims = [1, 0] x [1, 0], precision = [DEFAULT, DEFAULT] : (tensor<2x7xf64>

In [19]:
@code_xla contract(tnre)

HloModule reactant_contract, is_scheduled=true, entry_computation_layout={(f64[7,6,7,8,5,2]{5,4,3,2,1,0}, f64[6,7]{1,0}, f64[2,7]{1,0}, f64[5,8]{1,0})->f64[]}

%fused_computation (param_0.1: f64[14], param_1.7: f64[2,7]) -> f64[] {
  %param_1.7 = f64[2,7]{1,0} parameter(1)
  %transpose.9 = f64[7,2]{0,1} transpose(%param_1.7), dimensions={1,0}
  %copy.6 = f64[7,2]{1,0} copy(%transpose.9)
  %bitcast.10 = f64[14]{0} bitcast(%copy.6)
  %param_0.1 = f64[14]{0} parameter(0)
  %bitcast.9 = f64[2,7]{1,0} bitcast(%param_0.1)
  %transpose.8 = f64[7,2]{0,1} transpose(%bitcast.9), dimensions={1,0}
  %copy.5 = f64[7,2]{1,0} copy(%transpose.8)
  %bitcast.8 = f64[14]{0} bitcast(%copy.5)
  ROOT %dot.6 = f64[] dot(%bitcast.10, %bitcast.8), lhs_contracting_dims={0}, rhs_contracting_dims={0}
}

%fused_computation.1 (param_0.5: f64[5,8], param_1.13: f64[560]) -> f64[14] {
  %param_1.13 = f64[560]{0} parameter(1)
  %bitcast.13 = f64[2,5,8,7]{3,2,1,0} bitcast(%param_1.13)
  %transpose.10 = f64[2,7,5,8]{1,3,